In [1]:
import polars as pl
import pickle
import os, sys
from transformers import AutoTokenizer, AutoModel  

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from nlp_embeddings import eval_embeddings

In [2]:
disease_ot = pl.read_parquet("../data/disease/disease.parquet")
disease_ot.head(1)

id,code,name,description,dbXRefs,parents,exactSynonyms,relatedSynonyms,narrowSynonyms,broadSynonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology,synonyms
str,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],struct[3],struct[4]
"""DOID_0050890""","""http://purl.obolibrary.org/obo…","""synucleinopathy""","""A neurodegenerative disease th…","[""MEDGEN:1682194"", ""MESH:D000080874"", … ""UMLS:C5191670""]","[""EFO_0005772"", ""MONDO_0021179""]","[""alpha Synucleinopathies"", ""synucleinopathy""]","[""alpha synucleinopathies"", ""synucleinopathies""]",[],[],[],[],"[""EFO_0006792"", ""EFO_1001050""]","[""EFO_0005772"", ""MONDO_0021179"", … ""OTAR_0000020""]","[""EFO_0000618"", ""OTAR_0000020""]","[""EFO_0006792"", ""EFO_1001050"", … ""MONDO_0014835""]","{false,false,{""http://purl.obolibrary.org/obo/DOID_0050890"",""DOID_0050890""}}","{[""alpha Synucleinopathies"", ""synucleinopathy""],[""alpha synucleinopathies"", ""synucleinopathies""],[],[]}"


In [3]:
# dbXRefs are always different from the diseaseId
(
    disease_ot
    .explode(pl.col("dbXRefs"))
    .select(["id","dbXRefs"])
    .with_columns((pl.col("id") == pl.col("dbXRefs")).alias("equal"))
)['equal'].any()

False

In [4]:
# name and synonyms
(
    disease_ot
    .explode(pl.col("exactSynonyms"))
    .select(["name","exactSynonyms"])
    .with_columns((pl.col("name") == pl.col("exactSynonyms")).alias("equal"))
)['equal'].any()

# True --> some synonyms are equal to the name

True

Data structure for vector embeddings storage v0.1
```
{
    Label   -> embedding
            -> Id
}
```

In [5]:
disease_names_df = (
    disease_ot
    .explode(pl.col("exactSynonyms"))
    .select(['id','name','exactSynonyms'])
    .unpivot(['name','exactSynonyms'], index="id")
    .unique(subset=['id','value'], keep='first', maintain_order=True)
    .filter(pl.col("value").is_not_null())
)
disease_names_df.head(2)

id,variable,value
str,str,str
"""DOID_0050890""","""name""","""synucleinopathy"""
"""DOID_10113""","""name""","""trypanosomiasis"""


In [6]:
disease_names_df['value'].is_null().any()

False

In [7]:
def load_model(model_name):
    with open("../models/"+model_name+".pkl","rb") as input:
        return pickle.load(input)

In [8]:
with open("../models/model_names.pkl","rb") as input:
    model_names = pickle.load(input)

print(model_names)

{'BioClinicalBERT': 'emilyalsentzer/Bio_ClinicalBERT', 'ClinicalBERT': 'medicalai/ClinicalBERT', 'BioBERT': 'dmis-lab/biobert-v1.1', 'BioClinical-ModernBERT': 'thomas-sounack/BioClinical-ModernBERT-base', 'SapBERT-from-PubMedBERT-fulltext': 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'}


In [9]:
embeddings_label = {}

#for model_name in ['']:

model_name = 'SapBERT-from-PubMedBERT-fulltext'
loaded_model = load_model(model_name)
model = loaded_model['model']
tokenizer = loaded_model['tokenizer']
input_texts = disease_names_df['value'].to_list()
embeddings_label[model_name] = eval_embeddings(
    model=model,
    tokenizer=tokenizer,
    input_texts=input_texts,
    pooling='mean',
    batch_size=32
)

cuda
Processing batches...
Using CLS representation...


  0%|          | 0/4210 [00:00<?, ?it/s]

In [27]:
disease_names_df.with_columns(
    pl.DataFrame({"embeddings": list(embeddings_label[model_name])})
)

id,variable,value,embeddings
str,str,str,"array[f32, 768]"
"""DOID_0050890""","""name""","""synucleinopathy""","[-0.982104, 0.099318, … 0.4078]"
"""DOID_10113""","""name""","""trypanosomiasis""","[-0.827013, -0.04076, … 0.280557]"
"""DOID_10718""","""name""","""giardiasis""","[-1.113425, -0.718677, … 0.430307]"
"""DOID_13406""","""name""","""pulmonary sarcoidosis""","[-0.752791, 0.054742, … 0.689332]"
"""DOID_1947""","""name""","""trichomoniasis""","[-0.336825, -1.087154, … 0.329123]"
…,…,…,…
"""Orphanet_99947""","""exactSynonyms""","""Charcot-Marie-Tooth neuropathy…","[-0.00119, -0.439948, … 0.523701]"
"""Orphanet_99947""","""exactSynonyms""","""HMSN IIA2""","[0.237155, -0.205999, … 0.744288]"
"""Orphanet_99947""","""exactSynonyms""","""HMSN2A2""","[0.570201, -0.147043, … 0.742883]"


In [28]:
disease_names_df.write_parquet("../output/id_value_embeddings.parquete", compression="zstd")